# Aquarium Detection — starter notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/data_science_practice/blob/main/website/public/modules/ms2a-machine-learning-practice/challenges/mlp-s6-aquarium-detection.ipynb)

Find every animal in an aquarium photo: a **bounding box** and a **class** for each, among
**7 classes** (fish, jellyfish, penguin, puffin, shark, starfish, stingray):
[challenge 195](https://ml-arena.com/viewchallenge/195) on ML-Arena. Metric: **mAP50-95**.

This notebook runs end-to-end: **download the data → look at the boxes → fine-tune a pretrained
YOLO → score it on held-out images → submit**.

> In Colab: **Runtime → Change runtime type → T4 GPU**, then run the cells top to bottom.
> The only thing you must edit is your API token (your **Profile** page on ml-arena.com).

## 0. Setup

In [ ]:
!pip install -q "mlarena-sdk>=3" ultralytics

import mlarena
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import random, shutil, sys, zipfile, torch
from pathlib import Path
from PIL import Image

API_TOKEN = "mlk_user_REPLACE_ME"   # <-- paste your token (Profile page)
CHALLENGE_ID = 195

client = mlarena.connect(api_key=API_TOKEN, base_url="https://ml-arena.com")

## 1. Get the data

`train.zip` holds 448 images and their labels, `test_images.zip` the 190 images you predict on.
`detection_metric.py` is the leaderboard's parser and metric: we import it to score locally.

In [ ]:
client.download_dataset(CHALLENGE_ID, "data/")
zipfile.ZipFile("data/train.zip").extractall("data/train")
zipfile.ZipFile("data/test_images.zip").extractall("data/test")
sys.path.insert(0, "data")
import detection_metric as dm

class_names = pd.read_csv("data/classes.csv")["name"].tolist()
train_ids = sorted(p.stem for p in Path("data/train/images").glob("*.jpg"))
test_images = sorted(Path("data/test/images").glob("*.jpg"))
print(len(train_ids), "training images,", len(test_images), "test images")
print(class_names)

## 2. Look at the boxes

A label file has one line per box: `class_id x_center y_center width height`, each coordinate
divided by the image's width or height (the **YOLO format**).

In [ ]:
def read_labels(image_id):
    lines = Path(f"data/train/labels/{image_id}.txt").read_text().split()
    return np.array(lines, dtype=float).reshape(-1, 5)

def draw(ax, image_path, boxes, title=""):
    img = Image.open(image_path)
    W, H = img.size
    ax.imshow(img)
    for cls, xc, yc, w, h in boxes[:, [0, -4, -3, -2, -1]]:
        color = plt.cm.tab10(int(cls))
        ax.add_patch(patches.Rectangle(((xc - w / 2) * W, (yc - h / 2) * H), w * W, h * H,
                                       linewidth=2, edgecolor=color, facecolor="none"))
        ax.text((xc - w / 2) * W, (yc - h / 2) * H, class_names[int(cls)], color="white",
                fontsize=7, bbox=dict(facecolor=color, pad=1, linewidth=0))
    ax.set_title(title, fontsize=9); ax.axis("off")

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, iid in zip(axes.flat, random.Random(0).sample(train_ids, 6)):
    boxes = read_labels(iid)
    draw(ax, f"data/train/images/{iid}.jpg", boxes, f"{len(boxes)} boxes")
plt.tight_layout(); plt.show()

counts = pd.Series(np.concatenate([read_labels(i)[:, 0] for i in train_ids]).astype(int))
print(counts.map(dict(enumerate(class_names))).value_counts())

**Questions**
- How many animals does an image hold? Which classes are rare, and why does that matter for a
  metric that averages over classes?
- Which boxes look hard: small, overlapping, cut by the image border?

## 3. Hold out a validation set

The test labels are private, so we keep 15 % of the training images aside to measure the model
before submitting, and write the `data.yaml` Ultralytics trains from.

In [ ]:
ids = train_ids[:]
random.Random(0).shuffle(ids)
split = {"val": ids[:len(ids) * 15 // 100], "train": ids[len(ids) * 15 // 100:]}
for name, part in split.items():
    for sub in ("images", "labels"):
        Path(f"yolo/{name}/{sub}").mkdir(parents=True, exist_ok=True)
    for iid in part:
        shutil.copy(f"data/train/images/{iid}.jpg", f"yolo/{name}/images/")
        shutil.copy(f"data/train/labels/{iid}.txt", f"yolo/{name}/labels/")
Path("aquarium.yaml").write_text(
    f"path: {Path('yolo').resolve()}\ntrain: train/images\nval: val/images\nnames: {class_names}\n")
print({k: len(v) for k, v in split.items()})

## 4. Fine-tune a pretrained YOLO

`yolov8n.pt` is the smallest YOLOv8, pretrained on COCO (80 everyday classes, no starfish).
Fine-tuning keeps its backbone and adapts the detection head to our 7 classes.

In [ ]:
from ultralytics import YOLO

device = 0 if torch.cuda.is_available() else "cpu"
model = YOLO("yolov8n.pt")
model.train(data="aquarium.yaml", epochs=30, imgsz=640, batch=16, seed=0, device=device)
model = YOLO(Path(model.trainer.save_dir) / "weights" / "best.pt")   # best validation epoch

## 5. Score it with the leaderboard's metric

For mAP, predict with a **low confidence threshold** (0.001): low-confidence boxes cost nothing
at the top of the ranking and add recall at the bottom.

In [ ]:
def detect(paths):
    out = {}
    for i in range(0, len(paths), 16):
        batch = [str(p) for p in paths[i:i + 16]]
        for p, r in zip(paths[i:i + 16], model.predict(batch, conf=0.001, device=device, verbose=False)):
            b = r.boxes   # class, confidence, box normalised as in the labels
            out[Path(p).stem] = np.column_stack([b.cls.cpu().numpy(), b.conf.cpu().numpy(), b.xywhn.cpu().numpy()]).reshape(-1, 6)
    return out

val_pred = detect(sorted(Path("yolo/val/images").glob("*.jpg")))
val_gt = {iid: read_labels(iid) for iid in split["val"]}
scores = dm.evaluate(val_gt, val_pred)
print("validation mAP50-95 %.4f   mAP50 %.4f" % (scores["map50_95"], scores["map50"]))
print({c: round(scores[f"ap_{c}"], 3) for c in class_names})

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for col, iid in enumerate(split["val"][:3]):
    draw(axes[0, col], f"yolo/val/images/{iid}.jpg", val_gt[iid], "ground truth")
    shown = val_pred[iid][val_pred[iid][:, 1] >= 0.25]   # draw the confident ones only
    draw(axes[1, col], f"yolo/val/images/{iid}.jpg", shown, "predicted, confidence >= 0.25")
plt.tight_layout(); plt.show()

**Questions**
- Which class has the lowest AP? Is it rare in the training set, small, or confused with another?
- What does the gap between mAP50 and mAP50-95 tell you about the boxes?

## 6. Predict the test images and submit

In [ ]:
test_pred = detect(test_images)
submission = pd.DataFrame({"image_id": list(test_pred),
                           "prediction_string": [dm.to_prediction_string(b) for b in test_pred.values()]})
submission.to_csv("submission.csv", index=False)
submission.head()

In [ ]:
result = client.submit(challenge_id=CHALLENGE_ID, files=["submission.csv"], submission_name="yolov8n-30ep")
print(result)

## 7. Where to go from here

Change one thing, compare the validation mAP50-95, submit only if it improved.

- **Train longer, or bigger.** `epochs=60`; `yolov8s.pt` or `yolo11s.pt` instead of the nano.
  Bigger models are more accurate and slower: is the gain worth it?
- **Image size.** `imgsz=800` or `1024` (the originals were 1024 px) helps small fish, at a cost
  in memory: lower `batch` if the GPU runs out.
- **Augmentation.** Ultralytics already applies mosaic, flips and colour jitter: look at
  `train_batch0.jpg` in the run folder, then try `degrees`, `scale` or `mixup`.
- **Use all the data.** Once the settings are chosen, retrain on all 448 images before the final
  submission.
- **Look at the errors.** Draw the validation images with the lowest recall: missed small
  objects, duplicate boxes, one class taken for another?